# KíLín — literature-project evaluation notebook

**Routing, conversational scope, retrieval, scientific evidence, and model
comparisons**

This notebook creates an independent literature workspace from the five PDFs in
`tests/fixtures`, exercises KíLín through its normal `RAGSession` API, and
records both *expected* and *observed* behavior. It is an **evaluation
notebook**, not an automated scientific-correctness judge.

### What this experiment distinguishes

1. **Intent routing:** was the configured intent selected, and did it invoke
   the expected tool?
2. **Conversation continuity:** did the router select the appropriate earlier
   turn(s) and preserve the document or result-list scope?
3. **Execution:** did retrieval or the selected tool produce a complete answer
   and valid source references?
4. **Scientific quality:** is the answer actually faithful to the papers?
   This requires inspection; a correct route is *not* proof of a correct answer.
5. **Model dependence:** when the documents and questions are held fixed, how
   do different answering or routing models behave?

### How to run it

Execute the notebook **one section at a time, in order**. Each test prints one
small diagnostic table; the full answer and router information are collapsed
beneath it. Use `inspect_case("CASE_ID")` for an exact, detailed record. At
the end, inspect the per-intent coverage, compare models, and export the
observations. Tool-heavy tests, report generation, and alternative-model
comparisons are **opt-in**.

**Important:** a saved trace can contain complete text from private documents,
answers, and model reasoning. All output remains in the local project directory.
The notebook does not submit external network lookups and does not change KíLín
source code or project-type configuration.

## 1. Experiment configuration

Edit this cell first. The **primary answering model** is deliberately separated
from the **router model**, so comparisons can change one variable at a time.
`use_chat_model()` is used after session construction; it does not edit
`profile.ini`. An initial project-default model load may still occur during
session creation.

Use a new `EXPERIMENT_ID` for each independent run. The notebook refuses to
overwrite an existing project. Model downloads are disabled by default: make
sure any selected MLX models are available locally.

**Cost controls:** quick tool plans are used by default, network plans are
cancelled, and the long reference/review/report cases are gated separately.

In [1]:
from datetime import datetime
from pathlib import Path

# The run directory is new; set a fixed ID to make a repeat reproducible.
EXPERIMENT_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
PROJECT_NAME = f"trial_literature_routing_{EXPERIMENT_ID}"
ALLOW_DOWNLOAD = False

MODEL_IDS = {
    "qwen8b": "Qwen/Qwen3-8B-MLX-4bit",
    "qwen14b": "Qwen/Qwen3-14B-MLX-4bit",
    "qwen38_27b": "lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly",
}
PRIMARY_MODEL = "qwen8b"
ROUTER_MODEL = "Qwen/Qwen3-4B-MLX-4bit"
COMPARISON_MODELS = ("qwen8b", "qwen38_27b")
COMPARISON_ROUTERS = (
    "Qwen/Qwen3-4B-MLX-4bit",
    "Qwen/Qwen3-8B-MLX-4bit",
)

RETRIEVAL_COUNT = 10
RUN_LONG_TOOLS = False
RUN_REPORTS = False
RUN_MODEL_COMPARISON = False
RUN_ROUTER_COMPARISON = False
RUN_GUI_EXPORT_TEST = False

# Inline answers stay compact; expanded details show complete responses.

assert PRIMARY_MODEL in MODEL_IDS
assert all(name in MODEL_IDS for name in COMPARISON_MODELS)
assert EXPERIMENT_ID.strip()

repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file():
    if repo_root == repo_root.parent:
        raise RuntimeError("Run this notebook inside the KíLín repository.")
    repo_root = repo_root.parent

project_dir = repo_root / "sandbox" / PROJECT_NAME
print("Repository:", repo_root)
print("Experiment:", PROJECT_NAME)
print("Project path:", project_dir)
print("Answer model:", MODEL_IDS[PRIMARY_MODEL])
print("Router model:", ROUTER_MODEL)

Repository: /Users/giovannivolpe/Documents/GitHub/KiLin
Experiment: trial_literature_routing_20261102-135028
Project path: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature_routing_20261102-135028
Answer model: Qwen/Qwen3-8B-MLX-4bit
Router model: Qwen/Qwen3-4B-MLX-4bit


## 2. Evaluation and inspection helpers

The helpers read **saved `ConversationTurn.parameters`** as the authoritative
source for intent, tool, history IDs, evidence, and token totals. They also
inspect the native trace's `intent.classification.decided` operation to report
shortcuts, fallbacks, the router's backward-reference flag, and discarded links.

There are two distinct test modes:

- **`run_case` — full pipeline:** routes, retrieves or invokes a tool, saves the
  answer, and enables subsequent tests to refer to it. This is the main
  end-to-end contract test.
- **`probe_route` — routing only:** calls KíLín's routing function on the
  *actual previous turns* without executing a tool or adding a conversation
  turn. This is a useful diagnostic but uses the current private
  `RAGSession._router_chat()` accessor and is not itself an end-to-end test.

A `MATCH` means the **specified routing/context expectations** were satisfied.
It does not mean that a scientific assertion, citation, or quotation was
validated. `REVIEW` identifies an observed mismatch, while `ERROR` identifies
a failed operation. `---` in the expected-routing or status column means
that no single expected intent was specified: the route is shown for
inspection, not counted as a `MATCH`. The compact inline answer is
shortened, but **Answer and routing details** contains the complete saved
response without a character limit. No test silently changes the router's
proposed intent. In the tool column, `---` means that no tool was
selected; `model` is displayed only when it is the actual tool name.


In [2]:
import csv
import html
import json
from contextlib import contextmanager
from time import perf_counter

from IPython.display import HTML, display

from kilin.answering import route_question
from kilin.conversations import ConversationStore
from kilin.project_type import project_type
from kilin.rag import RAGSession

CASE_RESULTS = []
CASE_INDEX = {}
CONVERSATION_INDEX = {}


def literature_intents():
    """Return configured literature intents without hard-coded tool names.

    Returns
    -------
    dict
        Current name-to-configuration mapping from literature.json.

    """
    path = repo_root / "kilin" / "project_types" / "literature.json"
    with path.open(encoding="utf-8") as stream:
        return json.load(stream)["intents"]


def expected_tool(intent):
    """Return the tool configured for an intent, or None for model answers.

    Parameters
    ----------
    intent : str or None
        Intent identifier in the current literature project type.

    Returns
    -------
    str or None
        Configured tool name, if present.

    """
    return literature_intents().get(intent, {}).get("tool", {}).get("name")


def _source_names(evidence):
    """Identify filenames in saved source evidence, retaining order.

    Parameters
    ----------
    evidence : sequence
        Evidence objects persisted by RAGSession.

    Returns
    -------
    list of str
        Unique document basenames.

    """
    found = []
    for item in evidence:
        path = getattr(item, "path", None)
        if path is None and isinstance(item, dict):
            path = item.get("path") or item.get("source")
        if path:
            name = Path(str(path)).name
            if name not in found:
                found.append(name)
    return found


def _trace_operation(stages, stage_name, suffix):
    """Locate an operation by semantic suffix, not sequence number.

    Parameters
    ----------
    stages : dict
        One native trace turn's stages.
    stage_name : str
        Stage name such as ``01_question``.
    suffix : str
        End of the operation name, without its sequence prefix.

    Returns
    -------
    dict
        Matching operation or an empty mapping.

    """
    operations = stages.get(stage_name, {}).get("operations", {})
    return next(
        (value for key, value in operations.items() if key.endswith(suffix)),
        {},
    )


def _native_route(path, turn_number):
    """Read routing and timing details from one saved native trace turn.

    Parameters
    ----------
    path : pathlib.Path
        Conversation trace path.
    turn_number : int
        One-based turn number in this conversation.

    Returns
    -------
    dict
        Router, history, retrieval, and model-call diagnostics.

    """
    try:
        with path.open(encoding="utf-8") as stream:
            trace = json.load(stream)
        turns = trace["sessions"]["05_conversations"]["turns"]
        stages = list(turns.values())[turn_number - 1]["stages"]
    except (OSError, ValueError, KeyError, IndexError, TypeError):
        return {"trace_available": False}

    routed = _trace_operation(
        stages, "01_question", "intent_classification_decided"
    ).get("details", {})
    history = _trace_operation(
        stages, "01_question", "history_selection_result"
    ).get("details", {})
    retrieval = _trace_operation(
        stages, "03_retrieval", "retrieval_result"
    ).get("details", {})
    settings = _trace_operation(
        stages, "01_question", "turn_settings_snapshot"
    ).get("details", {})
    model_calls = []
    for stage in stages.values():
        for name, value in stage.get("operations", {}).items():
            if name.endswith("llm_call_output"):
                model_calls.append(value.get("details", {}))
    return {
        "trace_available": True,
        "classification": routed.get("classification", {}),
        "shortcut": routed.get("shortcut"),
        "fallback": routed.get("fallback"),
        "dropped_turns": routed.get("dropped_turns", []),
        "router_raw": routed.get("raw"),
        "history": history,
        "retrieval": retrieval,
        "settings": settings,
        "model_calls": model_calls,
    }

/Users/giovannivolpe/Documents/GitHub/KiLin/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def _gui_callbacks(events, choice="default", confirmation="quick"):
    """Create non-networked callbacks and record each scripted decision.

    Parameters
    ----------
    events : list
        Mutable list populated with GUI-equivalent interactions.
    choice : str or tuple, optional
        ``default``, ``first20``, ``save_all``, ``cancel``, or a tuple
        ``("document", filename)``. A value of ``none`` omits the adapter.
    confirmation : str, optional
        ``quick``, ``continue``, or ``cancel`` for model plans. Network plans
        are always cancelled.

    Returns
    -------
    tuple
        Choice callback and confirmation callback for ``RAGSession.ask``.

    """

    def choose(question, options):
        selected = None
        if isinstance(choice, tuple) and choice[0] == "document":
            target = choice[1].casefold()
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if item.split(" (", 1)[0].casefold() == target
                ),
                None,
            )
        elif choice in ("default", "first20"):
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if item == "Show the first 20 here"
                ),
                None,
            )
            if selected is None and choice == "default" and options:
                if "which paper" in question.casefold() or (
                    "which document" in question.casefold()
                ):
                    selected = 0
        elif choice == "save_all":
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if item.startswith("Save all ")
                    and item.endswith(" to outputs/ as Markdown")
                ),
                None,
            )
        events.append(
            {
                "kind": "choice",
                "question": question,
                "options": list(options),
                "selected": selected,
            }
        )
        return selected

    def confirm(plan):
        decision = (
            "cancel"
            if getattr(plan, "operation", "model") == "network"
            else confirmation
        )
        events.append(
            {
                "kind": "confirmation",
                "tool": plan.tool,
                "operation": getattr(plan, "operation", "model"),
                "summary": plan.summary,
                "items": plan.items,
                "batches": plan.batches_to_run,
                "quick_items": plan.quick_items,
                "selected": decision,
            }
        )
        return decision

    choice_callback = None if choice == "none" else choose
    return choice_callback, confirm


@contextmanager
def conversation(label, *, model=None, router=None, branch_from=None):
    """Open an independent test conversation and close it reliably.

    Parameters
    ----------
    label : str
        Human-readable scenario group name.
    model : str or None, optional
        Key in MODEL_IDS; defaults to PRIMARY_MODEL.
    router : str or None, optional
        Router model identifier; defaults to ROUTER_MODEL.
    branch_from : str or None, optional
        Existing conversation ID whose saved history is cloned into this
        conversation. A new independent conversation is the default.

    Yields
    ------
    dict
        Active session, trace path, and case-to-turn registry.

    """
    selected = model or PRIMARY_MODEL
    if selected not in MODEL_IDS:
        raise ValueError(f"Unknown model key: {selected}")
    if label in CONVERSATION_INDEX:
        raise ValueError(f"Conversation label already used: {label}")

    store = ConversationStore(project_dir)
    if branch_from is not None:
        store.ensure()
    info = store.create(branch_from=branch_from)
    path = store.state_path(info)
    session = None
    try:
        session = RAGSession(
            project_dir,
            conversation_path=path,
            router_model=router or ROUTER_MODEL,
        )
        session.use_chat_model(MODEL_IDS[selected])
        context = {
            "label": label,
            "store": store,
            "info": info,
            "path": path,
            "session": session,
            "case_turn_ids": {},
            "model": selected,
            "router": router or ROUTER_MODEL,
            "parent": branch_from,
        }
        CONVERSATION_INDEX[label] = {
            "path": str(path.relative_to(project_dir)),
            "model": selected,
            "router": router or ROUTER_MODEL,
            "parent": branch_from,
        }
        print(f"Conversation {label}: {path.name}")
        print("  Answering:", session.chat_session.model_id)
        print("  Router:", session.router_model)
        yield context
    finally:
        if session is not None:
            session.close()


def _expected_links(context, case_ids):
    """Resolve named earlier cases to one-based conversation turn IDs.

    Parameters
    ----------
    context : dict
        Active conversation context.
    case_ids : tuple
        Previously run case identifiers.

    Returns
    -------
    list of int
        One-based expected turn numbers.

    """
    missing = [
        case_id
        for case_id in case_ids
        if case_id not in context["case_turn_ids"]
    ]
    if missing:
        raise ValueError(f"Missing earlier cases: {missing}")
    return [context["case_turn_ids"][item] for item in case_ids]


def _check_record(record):
    """Evaluate only declared routing and context contracts.

    Parameters
    ----------
    record : dict
        Recorded case with expected and actual routing fields.

    Returns
    -------
    list of str
        Contract mismatches requiring review.

    """
    problems = []
    expected = record.get("expected_intent")
    if expected is not None and record.get("intent") != expected:
        problems.append("intent")
    if record["mode"] == "full" and expected is not None:
        if record.get("tool") != record.get("expected_tool"):
            problems.append("tool")
    standalone = record.get("expected_standalone")
    if standalone is not None and record.get("standalone") != standalone:
        problems.append("standalone")
    actual = set(record.get("related_turn_ids", []))
    required = set(record.get("expected_related_ids", []))
    forbidden = set(record.get("forbidden_related_ids", []))
    if not required.issubset(actual):
        problems.append("missing prior-turn link")
    if forbidden & actual:
        problems.append("unrelated prior-turn link")
    if record.get("expected_retrieval") is not None:
        if record.get("retrieval_mode") != record["expected_retrieval"]:
            problems.append("retrieval policy")
    sources = set(record.get("sources", []))
    if record.get("expected_sources"):
        if not set(record["expected_sources"]).issubset(sources):
            problems.append("missing source")
    if set(record.get("forbidden_sources", [])) & sources:
        problems.append("out-of-scope source")
    if record.get("intent") in record.get("disallowed_intents", []):
        problems.append("disallowed intent")
    if record.get("invalid_citations"):
        problems.append("invalid citation identifiers")
    if record.get("completion"):
        problems.append("incomplete answer")
    return problems

In [4]:
def _brief(value, limit=95):
    """Return short escaped text for a compact diagnostic table.

    Parameters
    ----------
    value : object
        Value to show.
    limit : int, optional
        Maximum characters before truncation.

    Returns
    -------
    str
        Escaped, whitespace-normalized label.

    """
    text = " ".join(str(value if value is not None else "-").split())
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return html.escape(text)


def show_case(record):
    """Show one compact table and optional full answer diagnostics.

    Parameters
    ----------
    record : dict
        Recorded routing-only or end-to-end test case.

    """
    expected = record.get("expected_intent") or "---"
    actual = record.get("intent") or "-"
    tool = record.get("tool") or "---"
    if record["mode"] == "route-only":
        tool = "would: " + (expected_tool(record.get("intent")) or "---")
    else:
        wanted = record.get("expected_tool") or "---"
        tool = f"{wanted} → {tool}"
    related = record.get("related_turn_ids", [])
    need = record.get("expected_related_ids", [])
    tokens = record.get("usage", {})
    token_text = (
        f"{tokens.get('input_tokens', 0)}/" f"{tokens.get('output_tokens', 0)}"
        if tokens
        else "-"
    )
    source_names = record.get("sources", [])
    source_text = f"{len(source_names)}: " + ", ".join(source_names)
    status = record["status"]
    body = "".join(
        "<td style='padding:4px 7px;border-bottom:1px solid #bbb'>"
        + value
        + "</td>"
        for value in (
            _brief(f"{expected} → {actual}", 55),
            _brief(tool, 22),
            _brief(f"need {need}; got {related}", 29),
            _brief(record.get("retrieval_mode") or "-", 12),
            _brief(source_text, 47),
            _brief(token_text, 22),
            _brief(f"{record.get('seconds', 0):.2f}s"),
            _brief(status, 18),
        )
    )
    title = (
        f"<strong>{_brief(record['id'])}</strong> · "
        f"{_brief(record['question'], 145)}"
    )
    header = "".join(
        f"<th style='padding:4px 7px;text-align:left'>"
        f"{html.escape(name)}</th>"
        for name in (
            "Expected → actual",
            "Tool",
            "Prior links",
            "Retrieval",
            "Evidence",
            "Tokens in/out",
            "Time",
            "Routing check",
        )
    )
    answer = record.get("answer") or ""
    one_line_answer = _brief(answer, 180)
    issues = ", ".join(record.get("issues", [])) or "none"
    origin = f"{record.get('label', '?')} / {record.get('model', '?')}"
    trace_path = record.get("trace_path", "-")
    route = record.get("trace", {})
    model_calls = route.get("model_calls", [])
    limit_hits = sum(
        call.get("finish_reason") == "length" for call in model_calls
    )
    prior_turns = record.get("previous", [])
    chosen = set(record.get("related_turn_ids", []))
    prior_html = "".join(
        f"<li>{'*' if item['turn'] in chosen else ' '} "
        f"#{item['turn']} {_brief(item['question'], 100)}: "
        f"{_brief(item['summary'], 120)}</li>"
        for item in prior_turns
    )
    refers_back = route.get("classification", {}).get(
        "refers_back", route.get("refers_back")
    )
    details = (
        f"<div><b>Scenario:</b> {_brief(origin)} · "
        f"<b>Turn:</b> {_brief(record.get('turn_number'))} · "
        f"<b>Trace:</b> {_brief(trace_path, 130)}</div>"
        f"<div><b>Issues:</b> {_brief(issues, 240)}</div>"
        f"<div><b>Shortcut:</b> {_brief(route.get('shortcut'))} · "
        f"<b>Fallback:</b> {_brief(route.get('fallback'))}</div>"
        f"<div><b>History IDs:</b> "
        f"{_brief(record.get('history_turn_ids', []))} · "
        f"<b>Refers back:</b> {_brief(refers_back)}</div>"
        f"<div><b>Model calls:</b> {len(model_calls)} · "
        f"<b>Length stops:</b> {limit_hits} · "
        f"<b>Finish:</b> {_brief(record.get('finish_reason'))}</div>"
        "<details><summary>Earlier answer snapshots "
        "(* means linked by router)</summary><ul>"
        f"{prior_html}</ul></details>"
        "<pre style='white-space:pre-wrap;max-height:300px;"
        "overflow:auto;font-size:12px'>"
        f"{html.escape(answer)}</pre>"
    )
    display(
        HTML(
            f"<div style='margin:8px 0 15px 0;font-size:12px'>"
            f"<div style='margin-bottom:5px'>{title}</div>"
            "<div style='overflow-x:auto'>"
            "<table style='border-collapse:collapse;width:100%;"
            "font-size:11px'><thead><tr>"
            f"{header}</tr></thead><tbody><tr>{body}</tr></tbody></table>"
            "</div>"
            f"<div style='margin:3px 0'><b>Answer:</b> "
            f"{one_line_answer}</div>"
            f"<details style='margin-top:5px'><summary>Answer and routing "
            f"details</summary>{details}</details></div>"
        )
    )


def inspect_case(case_id, *, show_answer=True, show_prior=True):
    """Print the complete saved answer, related context, and trace data.

    Parameters
    ----------
    case_id : str
        ID shown by the compact diagnostic table.
    show_answer : bool, optional
        Whether to print the full answer, including any reasoning tags.
    show_prior : bool, optional
        Whether to print earlier linked questions and saved answer summaries.

    Returns
    -------
    dict
        Unmodified record collected by the evaluation harness.

    """
    record = CASE_INDEX[case_id]
    print(
        json.dumps(
            {
                key: value
                for key, value in record.items()
                if key not in ("answer", "trace")
            },
            indent=2,
            ensure_ascii=False,
        )
    )
    if show_prior:
        print("\nPREVIOUS TURN SNAPSHOT")
        for item in record.get("previous", []):
            marker = (
                "*"
                if item["turn"] in record.get("related_turn_ids", [])
                else " "
            )
            print(
                f"{marker} #{item['turn']}: {item['question']}\n"
                f"    {item['summary']}"
            )
    if show_answer:
        print("\nFULL ANSWER\n", record.get("answer", ""))
    print("\nNATIVE ROUTING DIAGNOSTICS")
    print(json.dumps(record.get("trace", {}), indent=2, ensure_ascii=False))
    return record

In [5]:
def _previous_snapshot(context):
    """Capture summaries of actual saved answers before the next case.

    Parameters
    ----------
    context : dict
        Active conversation state.

    Returns
    -------
    list of dict
        Prior questions, answer excerpts, and evidence sources.

    """
    snapshot = []
    for index, turn in enumerate(context["session"].turns, start=1):
        snapshot.append(
            {
                "turn": index,
                "question": turn.question,
                "summary": " ".join(turn.answer.split())[:210],
                "sources": _source_names(turn.evidence),
                "intent": turn.parameters.get("intent"),
                "tool": (turn.parameters.get("tool") or {}).get("name"),
            }
        )
    return snapshot


def _register(record):
    """Retain and display one result without changing any test outcome.

    Parameters
    ----------
    record : dict
        New unique case record.

    """
    if record["id"] in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {record['id']}")
    CASE_RESULTS.append(record)
    CASE_INDEX[record["id"]] = record
    show_case(record)


def run_case(
    context,
    case_id,
    question,
    *,
    intent=None,
    standalone=None,
    related_to=(),
    forbid_related=(),
    retrieval=None,
    sources=(),
    forbid_sources=(),
    disallow=(),
    choice="default",
    confirm="quick",
):
    """Run and record a full RAG turn with explicit context expectations.

    Parameters
    ----------
    context : dict
        Test conversation returned by ``conversation``.
    case_id : str
        Unique identifier used for later comparisons and follow-ups.
    question : str
        Unmodified user question.
    intent : str or None, optional
        Expected literature intent; None records an exploratory observation.
    standalone : bool or None, optional
        Expected standalone classification; None omits this check.
    related_to : tuple of str, optional
        Earlier case IDs whose turns must be linked by the router.
    forbid_related : tuple of str, optional
        Earlier case IDs the router should not link to this question.
    retrieval : str or None, optional
        Expected retrieval mode: search, reuse, document, or none.
    sources : tuple of str, optional
        Source filenames that must appear in saved evidence.
    forbid_sources : tuple of str, optional
        Source filenames that must not occur in saved evidence.
    disallow : tuple of str, optional
        Intents that must not be chosen, including ambiguous cases.
    choice : str or tuple, optional
        Scripted GUI document/list selection.
    confirm : str, optional
        GUI-equivalent confirmation mode, with network access disabled.

    Returns
    -------
    dict
        Recorded answer and machine-verifiable routing observations.

    Notes
    -----
    A routing mismatch is recorded without stopping the conversation.
    Later turns therefore test how the system handles imperfect history.
    Missing required previous turns is a test setup error and raises.

    """
    if intent is not None and intent not in literature_intents():
        raise ValueError(f"Unknown intent: {intent}")
    expected_links = _expected_links(context, related_to)
    forbidden_links = _expected_links(context, forbid_related)
    if case_id in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {case_id}")

    session = context["session"]
    before = len(session.turns)
    previous = _previous_snapshot(context)
    events = []
    choose, confirm_plan = _gui_callbacks(events, choice, confirm)
    started = perf_counter()
    record = {
        "id": case_id,
        "mode": "full",
        "label": context["label"],
        "model": session.chat_session.model_id,
        "router_model": session.router_model,
        "question": question,
        "expected_intent": intent,
        "expected_tool": expected_tool(intent),
        "expected_standalone": standalone,
        "expected_related_ids": expected_links,
        "forbidden_related_ids": forbidden_links,
        "expected_retrieval": retrieval,
        "expected_sources": list(sources),
        "forbidden_sources": list(forbid_sources),
        "disallowed_intents": list(disallow),
        "previous": previous,
        "trace_path": str(context["path"].relative_to(project_dir)),
        "gui_events": events,
    }
    try:
        answer = session.ask(
            question,
            retrieval_count=RETRIEVAL_COUNT,
            choice_callback=choose,
            confirm_callback=confirm_plan,
        )
        if len(session.turns) != before + 1:
            raise RuntimeError("Expected exactly one committed turn.")
        turn = session.turns[-1]
        parameters = turn.parameters
        tool = parameters.get("tool") or {}
        record.update(
            {
                "answer": answer,
                "turn_number": len(session.turns),
                "intent": parameters.get("intent"),
                "tool": tool.get("name"),
                "tool_status": tool.get("status"),
                "tool_summary": tool,
                "standalone": parameters.get("standalone"),
                "related_turn_ids": list(
                    parameters.get("related_turn_ids", [])
                ),
                "history_turn_ids": list(
                    parameters.get("history_turn_ids", [])
                ),
                "retrieval_mode": parameters.get("retrieval_mode"),
                "sources": _source_names(turn.evidence),
                "evidence_count": len(turn.evidence),
                "invalid_citations": parameters.get("invalid_citations", []),
                "completion": parameters.get("completion"),
                "usage": parameters.get("usage", {}),
                "finish_reason": parameters.get("finish_reason"),
                "trace": _native_route(context["path"], len(session.turns)),
            }
        )
        record["issues"] = _check_record(record)
        record["status"] = (
            ("REVIEW" if record["issues"] else "MATCH")
            if intent
            else ("REVIEW" if record["issues"] else "---")
        )
        context["case_turn_ids"][case_id] = len(session.turns)
    except Exception as error:
        record.update(
            {
                "answer": "",
                "intent": None,
                "sources": [],
                "status": "ERROR",
                "issues": [f"{type(error).__name__}: {error}"],
            }
        )
    record["seconds"] = round(perf_counter() - started, 3)
    _register(record)
    return record


def probe_route(
    context,
    case_id,
    question,
    *,
    intent=None,
    standalone=None,
    related_to=(),
    forbid_related=(),
    disallow=(),
):
    """Test routing against real conversation history without execution.

    Parameters
    ----------
    context : dict
        Active context with previously saved conversation turns.
    case_id : str
        Unique diagnostic case ID.
    question : str
        Question to route without saving or answering.
    intent : str or None, optional
        Intended router classification.
    standalone : bool or None, optional
        Expected independence from previous turns.
    related_to : tuple of str, optional
        Required previous case IDs.
    forbid_related : tuple of str, optional
        Previous case IDs that must not be linked.
    disallow : tuple of str, optional
        Intents that must not be selected for this request.

    Returns
    -------
    dict
        Routing-only decision and any diagnosed mismatches.

    Notes
    -----
    This uses KíLín's own ``route_question`` and the active session's
    private router accessor. It does not save an answer or execute a tool.

    """
    if intent is not None and intent not in literature_intents():
        raise ValueError(f"Unknown intent: {intent}")
    required = _expected_links(context, related_to)
    forbidden = _expected_links(context, forbid_related)
    if case_id in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {case_id}")

    session = context["session"]
    started = perf_counter()
    record = {
        "id": case_id,
        "mode": "route-only",
        "label": context["label"],
        "model": session.chat_session.model_id,
        "router_model": session.router_model,
        "question": question,
        "expected_intent": intent,
        "expected_tool": expected_tool(intent),
        "expected_standalone": standalone,
        "expected_related_ids": required,
        "forbidden_related_ids": forbidden,
        "disallowed_intents": list(disallow),
        "previous": _previous_snapshot(context),
        "trace_path": str(context["path"].relative_to(project_dir)),
    }
    try:
        config = project_type(session.project.profile.type_id)
        mark = session.usage_mark()
        decision = route_question(
            session._router_chat(), config, session.turns, question
        )
        record.update(
            {
                "intent": decision.intent,
                "tool": None,
                "standalone": decision.standalone,
                "related_turn_ids": list(decision.related_turn_ids),
                "retrieval_mode": config.intents[
                    decision.intent
                ].retrieval_mode,
                "sources": [],
                "answer": "(routing only: no answer)",
                "usage": session.usage_since(mark),
                "trace": {
                    "shortcut": decision.shortcut,
                    "fallback": decision.fallback,
                    "router_raw": decision.raw,
                    "dropped_turns": list(decision.dropped_turns),
                    "refers_back": decision.refers_back,
                },
            }
        )
        record["issues"] = _check_record(record)
        record["status"] = (
            ("REVIEW" if record["issues"] else "MATCH")
            if intent
            else ("REVIEW" if record["issues"] else "---")
        )
    except Exception as error:
        record.update(
            {
                "answer": "",
                "intent": None,
                "sources": [],
                "status": "ERROR",
                "issues": [f"{type(error).__name__}: {error}"],
            }
        )
    record["seconds"] = round(perf_counter() - started, 3)
    _register(record)
    return record

In [6]:
def show_results(*, prefix=None, group=None, mode=None):
    """Display a compact summary across selected recorded cases.

    Parameters
    ----------
    prefix : str or None, optional
        Limit case IDs to a prefix.
    group : str or None, optional
        Limit records to a named conversation group.
    mode : str or None, optional
        Limit to ``full`` or ``route-only`` cases.

    Returns
    -------
    list of dict
        Selected records.

    """
    rows = [
        record
        for record in CASE_RESULTS
        if (prefix is None or record["id"].startswith(prefix))
        and (group is None or record["label"] == group)
        and (mode is None or record["mode"] == mode)
    ]
    if not rows:
        print("No matching cases recorded.")
        return []
    headings = (
        "Case",
        "Model",
        "Expected",
        "Actual",
        "Links",
        "Result",
        "Seconds",
    )
    header = "".join(f"<th>{html.escape(name)}</th>" for name in headings)
    body = "".join(
        "<tr>"
        + "".join(
            f"<td>{_brief(value, 65)}</td>"
            for value in (
                item["id"],
                item.get("model", "-"),
                item.get("expected_intent") or "---",
                item.get("intent") or "-",
                str(item.get("related_turn_ids", [])),
                item["status"],
                f"{item['seconds']:.2f}",
            )
        )
        + "</tr>"
        for item in rows
    )
    display(
        HTML(
            "<div style='overflow-x:auto;font-size:12px'>"
            "<table style='border-collapse:collapse;width:100%'>"
            f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody>"
            "</table></div>"
        )
    )
    return rows


def show_coverage():
    """Report coverage against all configured literature intents.

    Returns
    -------
    dict
        Intent-to-observation mapping for the active experiment.

    """
    coverage = {}
    for intent, settings in literature_intents().items():
        records = [
            item for item in CASE_RESULTS if item["expected_intent"] == intent
        ]
        statuses = [item["status"] for item in records]
        coverage[intent] = {
            "tool": settings.get("tool", {}).get("name"),
            "full": sum(item["mode"] == "full" for item in records),
            "probes": sum(item["mode"] == "route-only" for item in records),
            "matched": statuses.count("MATCH"),
            "review": statuses.count("REVIEW"),
            "errors": statuses.count("ERROR"),
        }
    heading = "".join(
        f"<th>{html.escape(name)}</th>"
        for name in (
            "Intent",
            "Expected tool",
            "Full",
            "Probes",
            "Match",
            "Review",
            "Errors",
        )
    )
    cells = "".join(
        "<tr>"
        + "".join(
            f"<td>{_brief(value)}</td>"
            for value in (
                intent,
                item["tool"] or "---",
                item["full"],
                item["probes"],
                item["matched"],
                item["review"],
                item["errors"],
            )
        )
        + "</tr>"
        for intent, item in coverage.items()
    )
    display(
        HTML(
            "<div style='overflow-x:auto;font-size:12px'>"
            "<table style='border-collapse:collapse;width:100%'>"
            f"<thead><tr>{heading}</tr></thead><tbody>{cells}</tbody>"
            "</table></div>"
        )
    )
    return coverage


def show_answer_model_comparison():
    """Compare the same numbered prompts across answering models.

    Returns
    -------
    list of dict
        Matching full-pipeline answering-model records.

    """
    items = [item for item in CASE_RESULTS if item["id"].startswith("M_")]
    if not items:
        print("No answering-model comparison has been run.")
        return []
    items = sorted(
        items, key=lambda item: (item["id"].rsplit("_", 1)[-1], item["model"])
    )
    headers = (
        "Prompt",
        "Answer model",
        "Intent",
        "Sources",
        "Seconds",
        "Answer excerpt",
        "Route",
    )
    head = "".join(f"<th>{html.escape(name)}</th>" for name in headers)
    rows = "".join(
        "<tr>"
        + "".join(
            f"<td>{_brief(value, 145)}</td>"
            for value in (
                item["id"].rsplit("_", 1)[-1],
                item["model"],
                item.get("intent"),
                ", ".join(item.get("sources", [])),
                f"{item['seconds']:.2f}",
                item.get("answer", ""),
                item["status"],
            )
        )
        + "</tr>"
        for item in items
    )
    display(
        HTML(
            "<div style='overflow-x:auto;font-size:11px'>"
            "<table style='width:100%;border-collapse:collapse'>"
            f"<thead><tr>{head}</tr></thead><tbody>{rows}</tbody>"
            "</table></div>"
        )
    )
    return items


def annotate_case(case_id, *, scientific=None, note=""):
    """Attach a human scientific-quality verdict without model grading.

    Parameters
    ----------
    case_id : str
        Existing case ID.
    scientific : str or None, optional
        Manual rating such as ``correct``, ``mixed``, or ``incorrect``.
    note : str, optional
        Explain which source evidence was checked.

    """
    CASE_INDEX[case_id]["manual_review"] = {
        "scientific": scientific,
        "note": note,
    }


def export_evaluation():
    """Write local JSON/CSV records and the experiment manifest.

    Returns
    -------
    pathlib.Path
        Directory holding the saved evaluation files.

    """
    folder = project_dir / "outputs" / "notebook_evaluation"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "cases.json").write_text(
        json.dumps(CASE_RESULTS, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    with (folder / "cases.csv").open(
        "w", encoding="utf-8", newline=""
    ) as stream:
        columns = [
            "id",
            "mode",
            "label",
            "model",
            "router_model",
            "expected_intent",
            "intent",
            "expected_tool",
            "tool",
            "standalone",
            "related_turn_ids",
            "history_turn_ids",
            "retrieval_mode",
            "sources",
            "evidence_count",
            "status",
            "issues",
            "seconds",
            "trace_path",
        ]
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        for record in CASE_RESULTS:
            writer.writerow(
                {
                    name: (
                        json.dumps(record.get(name), ensure_ascii=False)
                        if isinstance(record.get(name), (list, dict))
                        else record.get(name)
                    )
                    for name in columns
                }
            )
    manifest = {
        "experiment_id": EXPERIMENT_ID,
        "project": str(project_dir),
        "primary_model": MODEL_IDS[PRIMARY_MODEL],
        "router_model": ROUTER_MODEL,
        "comparison_models": [MODEL_IDS[k] for k in COMPARISON_MODELS],
        "allow_download": ALLOW_DOWNLOAD,
        "retrieval_count": RETRIEVAL_COUNT,
        "document_hashes": globals().get("FIXTURE_HASHES", {}),
        "literature_intents": list(literature_intents()),
        "conversations": CONVERSATION_INDEX,
    }
    (folder / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print("Exported:", folder)
    return folder

### 2.1 Protocol preflight

Confirm that KíLín's installed literature configuration still contains the
expected **21 intents**. If it has changed, stop and update the tests instead
of silently omitting new behavior. The table also shows the expected tool
for each intent; this is read from the current repository rather than
reproduced in the notebook helpers.

In [7]:
PLANNED_INTENTS = {
    "small_talk",
    "question_answering",
    "compare_sources",
    "report_generation",
    "conversation_followup",
    "rephrase_previous",
    "project_inventory",
    "mention_listing",
    "refine_results",
    "source_context",
    "paper_overview",
    "suggest_reference",
    "topic_listing",
    "fact_finding",
    "document_reading",
    "mention_classification",
    "reference_check",
    "research_briefing",
    "popular_science_summary",
    "technical_summary",
    "experimental_protocol",
}
configured = set(literature_intents())
assert PLANNED_INTENTS == configured, (
    "Literature intents changed. Review additions/removals before testing: "
    f"missing={sorted(PLANNED_INTENTS - configured)}, "
    f"new={sorted(configured - PLANNED_INTENTS)}"
)
print("Configured literature intents:", len(configured))
print("Full routing checks require comparing intent, tool and context.")
show_coverage()

Configured literature intents: 21
Full routing checks require comparing intent, tool and context.


Intent,Expected tool,Full,Probes,Match,Review,Errors
question_answering,---,0,0,0,0,0
fact_finding,find_fact,0,0,0,0,0
compare_sources,---,0,0,0,0,0
document_reading,batch_review,0,0,0,0,0
paper_overview,paper_overview,0,0,0,0,0
reference_check,reference_check,0,0,0,0,0
suggest_reference,suggest_reference,0,0,0,0,0
research_briefing,generate_paper_summary,0,0,0,0,0
popular_science_summary,generate_paper_summary,0,0,0,0,0
technical_summary,generate_paper_summary,0,0,0,0,0


{'question_answering': {'tool': None,
  'full': 0,
  'probes': 0,
  'matched': 0,
  'review': 0,
  'errors': 0},
 'fact_finding': {'tool': 'find_fact',
  'full': 0,
  'probes': 0,
  'matched': 0,
  'review': 0,
  'errors': 0},
 'compare_sources': {'tool': None,
  'full': 0,
  'probes': 0,
  'matched': 0,
  'review': 0,
  'errors': 0},
 'document_reading': {'tool': 'batch_review',
  'full': 0,
  'probes': 0,
  'matched': 0,
  'review': 0,
  'errors': 0},
 'paper_overview': {'tool': 'paper_overview',
  'full': 0,
  'probes': 0,
  'matched': 0,
  'review': 0,
  'errors': 0},
 'reference_check': {'tool': 'reference_check',
  'full': 0,
  'probes': 0,
  'matched': 0,
  'review': 0,
  'errors': 0},
 'suggest_reference': {'tool': 'suggest_reference',
  'full': 0,
  'probes': 0,
  'matched': 0,
  'review': 0,
  'errors': 0},
 'research_briefing': {'tool': 'generate_paper_summary',
  'full': 0,
  'probes': 0,
  'matched': 0,
  'review': 0,
  'errors': 0},
 'popular_science_summary': {'tool': 'g

## 3. Create a fresh, initially empty literature workspace

This establishes a clean baseline. No source files are added yet. Starting
empty is important: it tests whether the assistant correctly distinguishes
conversation/general knowledge from evidence-backed literature claims.

The directory is protected against overwriting. Rerunning this creation cell
requires choosing a new experiment ID in Section 1.

In [8]:
from kilin.workspace import prepare_sources_workspace

if project_dir.exists():
    raise FileExistsError(
        f"Existing experiment: {project_dir}. "
        "Choose a new EXPERIMENT_ID; never overwrite prior results."
    )
project = prepare_sources_workspace(
    [],
    "literature",
    workspace_directory=project_dir,
    settings={"allow_download": ALLOW_DOWNLOAD},
)
print("Created:", project.root)
print("Sources:", project.source)

Created: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature_routing_20261102-135028
Sources: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature_routing_20261102-135028/sources


### 3.1 Empty-project and unsupported-evidence tests

The six prompts below reproduce the original notebook's zero-source checks.
Expect **no fabricated project papers**. A greeting and a general-knowledge
question may be answered, but claims framed as project evidence must acknowledge
that no papers are indexed. Examine `invalid_citations` and the full text
rather than treating a correct route as scientific validation.

In [9]:
with conversation("empty-project") as ctx:
    run_case(
        ctx,
        "E01",
        "hello",
        intent="small_talk",
        standalone=True,
        retrieval="none",
    )
    run_case(
        ctx,
        "E02",
        "What can you help me with?",
        intent=None,
        standalone=True,
    )
    run_case(
        ctx,
        "E03",
        "What is Brownian motion?",
        intent=None,
        standalone=True,
    )
    run_case(
        ctx,
        "E04",
        "Which articles are available in this project?",
        intent="project_inventory",
        retrieval="none",
    )
    run_case(
        ctx,
        "E05",
        "Summarize the articles in this project.",
        intent=None,
    )
    run_case(
        ctx,
        "E06",
        "Find articles about neural networks in this project.",
        intent="topic_listing",
    )

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Conversation empty-project: conversation_2026-11-02T12-50-28.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
small_talk → small_talk,--- → ---,need []; got [],none,0:,116/3,2.75s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
--- → small_talk,--- → ---,need []; got [],none,0:,2244/32,6.78s,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
--- → question_answering,--- → ---,need []; got [],search,0:,2308/1352,54.31s,REVIEW


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
project_inventory → project_inventory,project_inventory → p…,need []; got [],none,0:,2208/14,5.34s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
--- → project_inventory,--- → project_invento…,need []; got [],none,0:,2243/14,5.47s,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
topic_listing → topic_listing,find_passages → find_…,need []; got [],none,0:,2727/50,7.79s,MATCH


In [10]:
show_results(prefix="E");

Case,Model,Expected,Actual,Links,Result,Seconds
E01,Qwen/Qwen3-8B-MLX-4bit,small_talk,small_talk,[],MATCH,2.75
E02,Qwen/Qwen3-8B-MLX-4bit,---,small_talk,[],---,6.78
E03,Qwen/Qwen3-8B-MLX-4bit,---,question_answering,[],REVIEW,54.31
E04,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,5.34
E05,Qwen/Qwen3-8B-MLX-4bit,---,project_inventory,[],---,5.47
E06,Qwen/Qwen3-8B-MLX-4bit,topic_listing,topic_listing,[],MATCH,7.79


## 4. Add the five reference papers

Use the **same five PDFs** as the earlier BRAPH/DIVA experiments:
`mijalkov17braph`, `tiunn26braph2`, `mijalkov25computational`,
`patil25deep`, and `yao25connecting`.

The source set is fixed deliberately. Do not add unrelated files between model
comparisons, because the index and retrieval scope would no longer be
controlled. The first index build can take time; later conversations reuse it.

In [11]:
from hashlib import sha256

from kilin.workspace import add_project_sources

FIXTURE_NAMES = (
    "mijalkov17braph.pdf",
    "mijalkov25computational.pdf",
    "patil25deep.pdf",
    "tiunn26braph2.pdf",
    "yao25connecting.pdf",
)
fixture_dir = repo_root / "tests" / "fixtures"
articles = [fixture_dir / name for name in FIXTURE_NAMES]
missing = [p.name for p in articles if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Required test PDFs missing: {missing}")
project = add_project_sources(project_dir, articles)
FIXTURE_HASHES = {
    article.name: sha256(article.read_bytes()).hexdigest()
    for article in articles
}
for article in articles:
    print(
        f"{article.name:30s} SHA-256 prefix: "
        f"{FIXTURE_HASHES[article.name][:16]}"
    )
print("Imported PDFs:", len(list(project.source.glob("*.pdf"))))

mijalkov17braph.pdf            SHA-256 prefix: 8cf54f05b91d2cf9
mijalkov25computational.pdf    SHA-256 prefix: 09b7f5f693c45a36
patil25deep.pdf                SHA-256 prefix: 54295cf647568baf
tiunn26braph2.pdf              SHA-256 prefix: 17e83c8ded0f6707
yao25connecting.pdf            SHA-256 prefix: fe7ee76ca19acc9f
Imported PDFs: 5


## 5. Direct routing and document discovery

Start with standalone prompts whose targets are unambiguous. This section
compares file inventory, exact word mentions, paper overviews, semantic
retrieval, and comparison. In particular:

- **`mention_listing`** asks whether an exact word occurs.
- **`topic_listing`** identifies passages/files *by meaning* and can be much
  more expensive.
- **`paper_overview`** summarizes one paper;
- **`compare_sources`** compares sources;
- **`question_answering`** explains their scientific content.

We first test the inexpensive routes end-to-end, and reserve a full semantic
scan for the opt-in long-tool section.

In [12]:
with conversation("direct-routes") as ctx:
    run_case(
        ctx,
        "D01",
        "Which articles are available in this project?",
        intent="project_inventory",
        standalone=True,
        retrieval="none",
    )
    run_case(
        ctx,
        "D02",
        "List the sentences in tiunn26braph2.pdf that mention BRAPH.",
        intent="mention_listing",
        standalone=True,
        retrieval="none",
        choice="first20",
    )
    run_case(
        ctx,
        "D03",
        "Give me a short overview of tiunn26braph2.pdf.",
        intent="paper_overview",
        standalone=True,
        retrieval="none",
    )
    run_case(
        ctx,
        "D04",
        "According to the BRAPH 2 paper, why was its " "software developed?",
        intent="question_answering",
        retrieval="search",
        sources=("tiunn26braph2.pdf",),
    )
    run_case(
        ctx,
        "D05",
        "Compare the methods of mijalkov17braph.pdf "
        "and tiunn26braph2.pdf, including their differences.",
        intent="compare_sources",
        retrieval="search",
    )

Conversation direct-routes: conversation_2026-11-02T12-53-38.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
project_inventory → project_inventory,project_inventory → p…,need []; got [],none,0:,2124/14,5.62s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
mention_listing → mention_listing,scan_and_list → scan_…,need []; got [],none,1: tiunn26braph2.pdf,2437/35,5.91s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
paper_overview → paper_overview,paper_overview → pape…,need []; got [],none,1: tiunn26braph2.pdf,8559/653,55.82s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
question_answering → question_answering,--- → ---,need []; got [],search,"5: tiunn26braph2.pdf, yao25connecting.pdf, mij…",6018/533,37.79s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
compare_sources → compare_sources,--- → ---,"need []; got [3, 4]",search,"4: mijalkov17braph.pdf, tiunn26braph2.pdf, yao…",38347/901,318.77s,MATCH


In [13]:
show_results(prefix="D");

Case,Model,Expected,Actual,Links,Result,Seconds
D01,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,5.62
D02,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,5.91
D03,Qwen/Qwen3-8B-MLX-4bit,paper_overview,paper_overview,[],MATCH,55.82
D04,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],MATCH,37.79
D05,Qwen/Qwen3-8B-MLX-4bit,compare_sources,compare_sources,"[3, 4]",MATCH,318.77


## 6. Multi-turn conversational scope

These are **real saved-answer chains**, not independent classification prompts.
The same `RAGSession` is used throughout each chain. After every turn, the
notebook records its full answer, source filenames, one-based turn ID, and
router's selected related/history IDs.

A context test checks whether a follow-up refers to the *right previous turn*.
This is separate from whether the answer's content is scientifically correct.
For example, asking for DIVA methods must not silently change the active
paper back to the BRAPH memory-capacity study.

### 6.1 Referents across papers: BRAPH 2 → 2017 BRAPH → DIVA

The six-turn sequence reproduces the earlier cross-paper contamination
failure. The final `probe_route` questions are **not saved**, so they all
inspect the same conversation history. Look for the right related turn and
unwanted links back to an obsolete paper.

In [14]:
with conversation("paper-identity") as ctx:
    run_case(
        ctx,
        "P01",
        "Summarize the BRAPH 2 article.",
        intent="paper_overview",
        standalone=True,
    )
    run_case(
        ctx,
        "P02",
        "What are its main limitations?",
        intent="question_answering",
        standalone=False,
        related_to=("P01",),
    )
    run_case(
        ctx,
        "P03",
        "How does it compare with the 2017 paper?",
        intent="compare_sources",
        standalone=False,
        related_to=("P01",),
    )
    run_case(
        ctx,
        "P04",
        "Now tell me about the deep learning article.",
        intent="paper_overview",
        standalone=False,
    )
    run_case(
        ctx,
        "P05",
        "What methods does it use?",
        intent="question_answering",
        standalone=False,
        related_to=("P04",),
        forbid_related=("P01",),
        sources=("patil25deep.pdf",),
        forbid_sources=("mijalkov25computational.pdf",),
    )
    run_case(
        ctx,
        "P06",
        "Compare it with the previous article.",
        intent="compare_sources",
        standalone=False,
        related_to=("P04",),
    )
    probe_route(
        ctx,
        "P07",
        "What did the DIVA paper use as input?",
        intent="fact_finding",
        standalone=False,
        related_to=("P04",),
    )
    probe_route(
        ctx,
        "P08",
        "Which paper did we discuss first?",
        intent="conversation_followup",
        standalone=False,
    )

Conversation paper-identity: conversation_2026-11-02T13-13-56.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
paper_overview → paper_overview,paper_overview → pape…,need []; got [],none,1: tiunn26braph2.pdf,8068/744,62.56s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
question_answering → question_answering,--- → ---,need [1]; got [],search,"4: yao25connecting.pdf, tiunn26braph2.pdf, mij…",5996/572,42.52s,REVIEW


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
compare_sources → compare_sources,--- → ---,"need [1]; got [1, 2]",search,"4: tiunn26braph2.pdf, mijalkov17braph.pdf, yao…",8204/1031,74.95s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
paper_overview → question_answering,paper_overview → ---,need []; got [],search,"4: tiunn26braph2.pdf, patil25deep.pdf, mijalko…",6040/729,45.62s,REVIEW


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
question_answering → question_answering,--- → ---,need [4]; got [],search,"4: mijalkov25computational.pdf, yao25connectin…",5911/665,42.76s,REVIEW


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
compare_sources → compare_sources,--- → ---,"need [4]; got [4, 5]",search,"4: tiunn26braph2.pdf, mijalkov25computational.…",7099/731,50.71s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
fact_finding → question_answering,would: ---,need [4]; got [],search,0:,2558/16,5.60s,REVIEW


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
conversation_followup → conversation_followup,would: ---,need []; got [1],reuse,0:,2555/23,5.73s,MATCH


In [15]:
show_results(prefix="P");

Case,Model,Expected,Actual,Links,Result,Seconds
P01,Qwen/Qwen3-8B-MLX-4bit,paper_overview,paper_overview,[],MATCH,62.56
P02,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],REVIEW,42.52
P03,Qwen/Qwen3-8B-MLX-4bit,compare_sources,compare_sources,"[1, 2]",MATCH,74.95
P04,Qwen/Qwen3-8B-MLX-4bit,paper_overview,question_answering,[],REVIEW,45.62
P05,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],REVIEW,42.76
P06,Qwen/Qwen3-8B-MLX-4bit,compare_sources,compare_sources,"[4, 5]",MATCH,50.71
P07,Qwen/Qwen3-8B-MLX-4bit,fact_finding,question_answering,[],REVIEW,5.60
P08,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,conversation_followup,[1],MATCH,5.73


### 6.2 Follow-up versus rephrase versus a fresh answer

A reference to *what the assistant just said* is different from a fresh
question about the article. The router exposes `standalone`, `refers_back`,
`related_turn_ids`, and `history_turn_ids`; examine them together.

In [16]:
with conversation("answer-continuity") as ctx:
    run_case(
        ctx,
        "C01",
        "Give me an overview of the BRAPH 2 paper.",
        intent="paper_overview",
        standalone=True,
    )
    run_case(
        ctx,
        "C02",
        "Summarize its main contributions and limitations.",
        intent="question_answering",
        standalone=False,
        related_to=("C01",),
    )
    run_case(
        ctx,
        "C03",
        "In your previous answer, what limitation " "did you mention first?",
        intent="conversation_followup",
        standalone=False,
        related_to=("C02",),
        retrieval="reuse",
    )
    run_case(
        ctx,
        "C04",
        "Rewrite that answer in two concise bullets "
        "without adding any new information.",
        intent="rephrase_previous",
        standalone=False,
        related_to=("C03",),
        retrieval="reuse",
    )
    probe_route(
        ctx,
        "C05",
        "How does the architecture work in BRAPH 2?",
        intent="question_answering",
    )
    probe_route(
        ctx,
        "C06",
        "Could you explain your second bullet?",
        intent="conversation_followup",
        standalone=False,
        related_to=("C04",),
    )

Conversation answer-continuity: conversation_2026-11-02T13-19-29.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
paper_overview → paper_overview,paper_overview → pape…,need []; got [],none,1: tiunn26braph2.pdf,8434/655,56.97s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
question_answering → paper_overview,--- → paper_overview,need [1]; got [],none,1: tiunn26braph2.pdf,8574/812,63.82s,REVIEW


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
conversation_followup → conversation_followup,--- → ---,need [2]; got [2],reuse,1: tiunn26braph2.pdf,9220/271,45.72s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
rephrase_previous → rephrase_previous,--- → ---,need [3]; got [2],reuse,1: tiunn26braph2.pdf,9067/1310,90.58s,REVIEW


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
question_answering → question_answering,would: ---,need []; got [],search,0:,2400/16,5.40s,MATCH


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check
conversation_followup → conversation_followup,would: ---,need [4]; got [4],reuse,0:,2396/23,5.57s,MATCH


In [17]:
show_results(prefix="C");

Case,Model,Expected,Actual,Links,Result,Seconds
C01,Qwen/Qwen3-8B-MLX-4bit,paper_overview,paper_overview,[],MATCH,56.97
C02,Qwen/Qwen3-8B-MLX-4bit,question_answering,paper_overview,[],REVIEW,63.82
C03,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,conversation_followup,[2],MATCH,45.72
C04,Qwen/Qwen3-8B-MLX-4bit,rephrase_previous,rephrase_previous,[2],REVIEW,90.58
C05,Qwen/Qwen3-8B-MLX-4bit,question_answering,question_answering,[],MATCH,5.40
C06,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,conversation_followup,[4],MATCH,5.57


## 7. Previous-result tools and source boundaries

This tests one of the most important observed defects: after listing matches
from *one PDF*, an instruction to filter **those results** must not silently
re-run an unrestricted scan over all five PDFs. It also distinguishes:

- `source_context` / `show_source`: show context around a **previously
  identified citation or sentence**;
- `topic_listing` / `find_passages`: **find new passages** by their meaning;
- `refine_results` / `filter_results`: **operate on previous results**, not
  an unrelated or wider result set.

The two source-context probes intentionally differ by whether there is a
valid previous source-bearing answer.

In [ ]:
with conversation("result-scoping") as ctx:
    run_case(
        ctx,
        "S01",
        "List the sentences in tiunn26braph2.pdf " "that mention BRAPH.",
        intent="mention_listing",
        standalone=True,
        choice="first20",
    )
    run_case(
        ctx,
        "S02",
        "Show the exact source text and surrounding "
        "context for the first cited sentence in that list.",
        intent="source_context",
        standalone=False,
        related_to=("S01",),
        sources=("tiunn26braph2.pdf",),
    )
    run_case(
        ctx,
        "S03",
        "Of the BRAPH mentions you listed, "
        "which describe software features rather than "
        "bibliographic citations?",
        intent="refine_results",
        standalone=False,
        related_to=("S01",),
        forbid_sources=(
            "mijalkov17braph.pdf",
            "mijalkov25computational.pdf",
            "patil25deep.pdf",
            "yao25connecting.pdf",
        ),
    )
    probe_route(
        ctx,
        "S04",
        "Filter those earlier results to descriptions "
        "of graphical user interfaces.",
        intent="refine_results",
        standalone=False,
        related_to=("S03",),
    )
    probe_route(
        ctx,
        "S05",
        "Find new passages in tiunn26braph2.pdf "
        "explaining how its graphical interface works.",
        intent="topic_listing",
        standalone=True,
    )
    probe_route(
        ctx,
        "S06",
        "Show the source context for the first " "sentence you just quoted.",
        intent="source_context",
        standalone=False,
        related_to=("S02",),
    )
show_results(prefix="S")

### 7.1 List-display, export, and noninteractive choices

The previous notebook tested three behavior paths for large mention lists.
First the assistant lists the files, so "these articles" has a real referent.
This keeps choices separate from semantic routing: inspect the recorded
`gui_events` in `inspect_case` to confirm which option was offered and used.
The export branch writes an artifact only when explicitly enabled.

In [ ]:
with conversation("mention-choices") as ctx:
    run_case(
        ctx,
        "G00",
        "Which articles are available in this project?",
        intent="project_inventory",
    )
    run_case(
        ctx,
        "G01",
        "Which of these articles discuss BRAPH?",
        intent="mention_listing",
        choice="first20",
    )
    run_case(
        ctx,
        "G02",
        "Which of these articles discuss BRAPH?",
        intent="mention_listing",
        choice="none",
    )
    if RUN_GUI_EXPORT_TEST:
        run_case(
            ctx,
            "G03",
            "Which of these articles discuss BRAPH?",
            intent="mention_listing",
            choice="save_all",
        )
    else:
        print("G03 omitted: RUN_GUI_EXPORT_TEST = False")
show_results(prefix="G")

## 8. Routing boundary probes (lightweight)

This group is intentionally broad but relatively inexpensive: it checks
what the router *would* select without invoking expensive specialized tools.
It is useful for prompt/intent-description regression, and for distinguishing
false-positive keyword matches from actual intended operations. Cases with
expected intent `---` have no predetermined correct route (for example,
classifying whole documents by topic); their routing is shown for review
rather than scored as a match.

**No answers are added by probes.** Their outcomes validate the routing
decision and prior-turn links only. Full-tool behavior is tested in later
sections.

In [ ]:
with conversation("routing-boundaries") as ctx:
    run_case(
        ctx,
        "B00",
        "Give me a short overview of tiunn26braph2.pdf.",
        intent="paper_overview",
        standalone=True,
    )
    run_case(
        ctx,
        "B00b",
        "List the sentences in tiunn26braph2.pdf " "that mention BRAPH.",
        intent="mention_listing",
        choice="first20",
    )
    probe_route(
        ctx,
        "B01",
        "Does tiunn26braph2.pdf contain the exact word " "'MATLAB'?",
        intent="mention_listing",
        standalone=True,
    )
    probe_route(
        ctx,
        "B02",
        "Find passages about the limitations of the "
        "architecture, even when the word limitations is absent.",
        intent="topic_listing",
    )
    probe_route(
        ctx,
        "B03",
        "Find the passage in the BRAPH 2 paper "
        "describing its software architecture.",
        intent="topic_listing",
    )
    probe_route(
        ctx,
        "B04",
        "Show me the context surrounding the first "
        "cited passage in your previous answer.",
        intent="source_context",
        standalone=False,
        related_to=("B00b",),
    )
    probe_route(
        ctx,
        "B05",
        "What programming language is BRAPH 2 " "implemented in?",
        intent="fact_finding",
    )
    probe_route(
        ctx,
        "B06",
        "Which papers use graph methods for "
        "brain-connectivity analysis, by meaning?",
        intent="topic_listing",
    )
    probe_route(
        ctx,
        "B07",
        "Classify the five documents by their " "main research topic.",
        intent=None,
    )
    probe_route(
        ctx,
        "B08",
        "Group the previously listed sentences "
        "into software descriptions and citations.",
        intent="refine_results",
        standalone=False,
        related_to=("B00b",),
    )
    probe_route(
        ctx,
        "B08b",
        "Classify every mention of BRAPH in "
        "tiunn26braph2.pdf as a software description or citation.",
        intent="mention_classification",
        standalone=True,
    )
    probe_route(
        ctx,
        "B09",
        "Check whether the BRAPH 2 paper states that "
        "it includes a graphical user interface.",
        intent=None,
        disallow=("reference_check",),
    )
    probe_route(
        ctx,
        "B10",
        "Audit the references cited by " "tiunn26braph2.pdf.",
        intent="reference_check",
    )
    probe_route(
        ctx,
        "B11",
        "Which paper should I cite for the "
        "introduction of the BRAPH framework?",
        intent="suggest_reference",
    )
    probe_route(
        ctx,
        "B12",
        "Read tiunn26braph2.pdf from start to "
        "finish and enumerate its components.",
        intent="document_reading",
    )
    probe_route(
        ctx,
        "B13",
        "Prepare a detailed technical summary of "
        "tiunn26braph2.pdf for specialists.",
        intent="technical_summary",
    )
    probe_route(
        ctx,
        "B13b",
        "Prepare a research briefing for " "tiunn26braph2.pdf.",
        intent="research_briefing",
    )
    probe_route(
        ctx,
        "B13c",
        "Write a popular science summary for "
        "tiunn26braph2.pdf for a general audience.",
        intent="popular_science_summary",
    )
    probe_route(
        ctx,
        "B13d",
        "Extract an experimental protocol from "
        "patil25deep.pdf, noting missing replication details.",
        intent="experimental_protocol",
    )
    probe_route(
        ctx,
        "B14",
        "How is BRAPH 2 different from the " "original BRAPH software?",
        intent="compare_sources",
    )
    probe_route(
        ctx,
        "B15",
        "Write a structured report comparing "
        "graph-analysis methods across these papers.",
        intent="report_generation",
    )
show_results(prefix="B")

## 9. Hard continuity cases from the BRAPH experiments

These cases recreate previously observed failures: an unexpected citation
audit instead of a claim check, a one-paper search instead of all five,
incorrectly interpreting DIVA as a brain-network study, and cross-document
mixing. The lightweight probes run first; the expensive semantic and audit
tools are left for Section 10.

In [ ]:
with conversation("hard-scope") as ctx:
    run_case(
        ctx,
        "H01",
        "How does BRAPH 2 differ from the " "earlier BRAPH framework?",
        intent="compare_sources",
        standalone=True,
    )
    probe_route(
        ctx,
        "H02",
        "What evidence supports those differences?",
        intent="conversation_followup",
        standalone=False,
        related_to=("H01",),
    )
    probe_route(
        ctx,
        "H03",
        "Check whether the BRAPH 2 paper explicitly "
        "supports the claim that it provides a GUI.",
        intent=None,
        disallow=("reference_check",),
    )
    probe_route(
        ctx,
        "H04",
        "List the datasets used by all five papers.",
        intent="fact_finding",
    )
    probe_route(
        ctx,
        "H05",
        "Do all five papers use the same dataset? " "Explain the differences.",
        intent="compare_sources",
    )
    probe_route(
        ctx,
        "H06",
        "Review the five articles for relevance to "
        "a literature review on computational brain connectivity.",
        intent=None,
        disallow=("reference_check",),
    )
show_results(prefix="H")

### 9.1 Source-specific scientific spot checks

The next three end-to-end questions test whether a correct routing decision
actually uses the relevant paper. Scientific interpretation must still be
reviewed against the PDF. Relevant checkpoints are: the Raman-stress study
uses **BRAPH2DiSTAP**, the original BRAPH paper was published in 2017, and
BRAPH 2 discusses MATLAB/Python interoperability as future development.

In the final question, requiring all five sources is intentional: if retrieved
evidence omits one, the test marks a scope limitation rather than allowing
an unsupported collection-wide conclusion.

In [ ]:
with conversation("scientific-scope") as ctx:
    run_case(
        ctx,
        "X01",
        "How is BRAPH2DiSTAP used in patil25deep.pdf, "
        "and what does the study measure?",
        intent="question_answering",
        retrieval="search",
        sources=("patil25deep.pdf",),
        forbid_sources=("mijalkov25computational.pdf",),
    )
    run_case(
        ctx,
        "X02",
        "Does tiunn26braph2.pdf report that direct "
        "Python integration is implemented or future work?",
        intent="question_answering",
        retrieval="search",
        sources=("tiunn26braph2.pdf",),
    )
    run_case(
        ctx,
        "X03",
        "Compare the datasets used in all five "
        "project papers. Are they the same dataset?",
        intent="compare_sources",
        retrieval="search",
        sources=FIXTURE_NAMES,
    )
show_results(prefix="X")

## 10. Full end-to-end tool and intent coverage (opt-in)

The earlier sections establish inexpensive routing and conversation behavior.
This section exercises the remaining expensive tools and report templates
**only when enabled**. Running everything with a large thinking model can
require many thousands of tokens and substantial wall time.

`quick` plans are chosen for model-intensive tools; this tests tool selection
and partial execution, **not** exhaustive processing. Network operations are
cancelled. A correct route does not imply a completed or accurate report.

### 10.1 Extensive scans, classification, fact extraction, and references

Use this to test the real `find_passages`, `find_fact`, `batch_review`,
`batch_classify`, `suggest_reference`, and `reference_check` operations.
Check generated passages and quotation verification manually.

In [ ]:
if RUN_LONG_TOOLS:
    with conversation("long-tools") as ctx:
        run_case(
            ctx,
            "L01",
            "Which papers apply graph-theory methods to "
            "analyze brain connectivity, based on meaning rather "
            "than keywords?",
            intent="topic_listing",
            confirm="quick",
        )
        run_case(
            ctx,
            "L02",
            "Give a complete list of the programming "
            "languages and software dependencies named in "
            "tiunn26braph2.pdf.",
            intent="fact_finding",
            confirm="quick",
        )
        run_case(
            ctx,
            "L03",
            "Read the whole tiunn26braph2.pdf paper "
            "from start to end and identify its components.",
            intent="document_reading",
            confirm="quick",
        )
        run_case(
            ctx,
            "L04",
            "Classify the mentions of BRAPH in "
            "tiunn26braph2.pdf into current software and citations.",
            intent="mention_classification",
            confirm="quick",
        )
        run_case(
            ctx,
            "L05",
            "Check the references cited by "
            "tiunn26braph2.pdf and determine which cited papers "
            "are present in this project.",
            intent="reference_check",
            confirm="quick",
        )
        run_case(
            ctx,
            "L06",
            "Which article should I cite for the "
            "introduction of the BRAPH framework?",
            intent="suggest_reference",
            confirm="quick",
        )
        run_case(
            ctx,
            "L07",
            "Suggest a reference from this project for "
            "the statement: BRAPH supports graph analysis of brain "
            "networks.",
            intent="suggest_reference",
            confirm="quick",
        )
        run_case(
            ctx,
            "L08",
            "Review the five articles for their "
            "relevance to computational brain connectivity.",
            intent="mention_classification",
            confirm="quick",
        )
else:
    print("Long-tool cases omitted: RUN_LONG_TOOLS = False")

### 10.2 Named reporting templates

The four report intents deliberately share `generate_paper_summary` but must
select different templates. The notebook records the selected intent, tool
summary (including template when present), saved output path, and native trace.
Review scientific specificity and fabricated method/benchmark claims in the
produced Markdown. No Qt PDF writer is supplied in notebook mode.

In [ ]:
if RUN_REPORTS:
    with conversation("report-templates") as ctx:
        run_case(
            ctx,
            "T01",
            "Prepare a research briefing for " "tiunn26braph2.pdf.",
            intent="research_briefing",
            confirm="quick",
        )
        run_case(
            ctx,
            "T02",
            "Write a popular science summary of "
            "tiunn26braph2.pdf for a general audience.",
            intent="popular_science_summary",
            confirm="quick",
        )
        run_case(
            ctx,
            "T03",
            "Prepare a detailed technical summary of "
            "tiunn26braph2.pdf for specialists.",
            intent="technical_summary",
            confirm="quick",
        )
        run_case(
            ctx,
            "T04",
            "Extract an experimental protocol from "
            "tiunn26braph2.pdf; explicitly identify missing "
            "replication details instead of inventing them.",
            intent="experimental_protocol",
            confirm="quick",
        )
else:
    print("Report templates omitted: RUN_REPORTS = False")

### 10.3 Explicit GUI cancellation and document selection

The tool callback should record **Cancel** on a confirmation plan without
proceeding to a long model run. For ambiguous paper selection, the notebook
selects `tiunn26braph2.pdf` by its exact offered filename rather than always
taking the first option.

In [ ]:
with conversation("gui-paths") as ctx:
    run_case(
        ctx,
        "U01",
        "Give me a short overview of one paper " "in this collection.",
        intent="paper_overview",
        choice=("document", "tiunn26braph2.pdf"),
    )
    run_case(
        ctx,
        "U02",
        "Give a complete list of the programming "
        "languages in tiunn26braph2.pdf.",
        intent="fact_finding",
        confirm="cancel",
    )
show_results(prefix="U")

## 11. Controlled answering-model comparison (opt-in)

The answering-model comparison holds the **five documents, router model,
questions, retrieval count, and project settings** fixed. Each answering model
gets a **fresh conversation**, preventing previous answers from one model
contaminating another. The three selected questions include an explicit
single-paper request, a two-paper comparison, and a context-dependent follow-up.

Only `PRIMARY_MODEL` and `COMPARISON_MODELS` are used. Models must already be
available locally when `ALLOW_DOWNLOAD=False`. Model switching is supported
for the MLX backend; changing from MLX to Torch requires a separate project
configuration, not simply `use_chat_model()`.

**Interpretation:** the router is fixed, but its related-turn decisions may
differ because each model's previous answers differ. Compare both the answer
and the recorded links, not just elapsed time.

In [ ]:
if RUN_MODEL_COMPARISON:
    for model_key in dict.fromkeys(COMPARISON_MODELS):
        group = f"answer-model-{model_key}"
        with conversation(group, model=model_key) as ctx:
            run_case(
                ctx,
                f"M_{model_key}_01",
                "Give me an overview of the DIVA deep-learning "
                "paper patil25deep.pdf.",
                intent="paper_overview",
                standalone=True,
            )
            run_case(
                ctx,
                f"M_{model_key}_02",
                "What methods does it use, and what is the input "
                "data modality?",
                intent="question_answering",
                standalone=False,
                related_to=(f"M_{model_key}_01",),
            )
            run_case(
                ctx,
                f"M_{model_key}_03",
                "Compare those methods with the BRAPH 2 "
                "software-framework paper tiunn26braph2.pdf.",
                intent="compare_sources",
                standalone=False,
                related_to=(f"M_{model_key}_02",),
            )
    show_answer_model_comparison()
else:
    print("Answering-model comparison omitted.")

## 12. Controlled router-model comparison (opt-in)

This comparison changes **only the router model**, with the same base
answering model and question wording. It uses *routing-only probes* to avoid
making the answer model perform expensive, irrelevant work.

One seed conversation is saved **once** under the default router. Its saved
history and source evidence are then **branched** into two independent
conversations using `ConversationStore.create(branch_from=...)`. Each router
therefore receives **the exact same prior answer and evidence**. Router-only
probes inspect the same questions against that identical history; they do not
add turns or run tools.

In [ ]:
if RUN_ROUTER_COMPARISON:
    with conversation("router-seed") as seed_ctx:
        run_case(
            seed_ctx,
            "Rseed00",
            "Give me a short overview of " "tiunn26braph2.pdf.",
            intent="paper_overview",
        )
        parent_id = seed_ctx["info"].conversation_id

    for index, router_id in enumerate(COMPARISON_ROUTERS, start=1):
        group = f"router-model-{index}"
        with conversation(
            group, router=router_id, branch_from=parent_id
        ) as ctx:
            if len(ctx["session"].turns) != 1:
                raise RuntimeError("Router branch did not restore seed turn.")
            ctx["case_turn_ids"]["Rseed00"] = 1
            probe_route(
                ctx,
                f"R{index}_01",
                "Explain the second contribution you mentioned.",
                intent="conversation_followup",
                standalone=False,
                related_to=("Rseed00",),
            )
            probe_route(
                ctx,
                f"R{index}_02",
                "Which papers contain the exact word 'BRAPH'?",
                intent="mention_listing",
                standalone=True,
            )
            probe_route(
                ctx,
                f"R{index}_03",
                "Which papers discuss brain connectivity, by meaning?",
                intent="topic_listing",
            )
            probe_route(
                ctx,
                f"R{index}_04",
                "Check whether that paper supports the claim "
                "that it provides a GUI.",
                intent=None,
                standalone=False,
                related_to=("Rseed00",),
                disallow=("reference_check",),
            )
    show_results(prefix="R")
else:
    print("Router-model comparison omitted.")

## 13. Review and export the evaluation

Read the complete routing table **after** completing the desired sections.
The table includes all 21 configured intents; an untested intent remains
explicitly untested rather than being counted as a success. Model comparisons
and routing probes are reported separately by `mode` and `model`.

For selected mismatches, call `inspect_case("S03")` (or another case ID) to
read the exact saved answer, all prior-answer snapshots, the actual router
JSON, selected turns, trace model calls, and GUI decisions. Use
`annotate_case()` to record manual scientific-quality checks. The exported
`cases.json` includes these notes; `cases.csv` is a concise overview.

**Do not interpret `MATCH` as a fact-check.** Scientific truth, quotations,
missing evidence, and whether citations substantiate a claim require a
separate human review of the source paper and the generated report.

In [ ]:
coverage = show_coverage()
all_end_to_end = show_results(mode="full")
all_routing = show_results(mode="route-only")
print("Total recorded cases:", len(CASE_RESULTS))
print("Unique conversations:", len(CONVERSATION_INDEX))

### 13.1 Inspect failures and disagreements

The next cell lists only cases that deserve attention. For example, after
inspecting a problematic answer, run `inspect_case("CASE_ID")` in a separate
cell. To preserve a scientific judgment in the export, use:

```python
annotate_case(
    "P05",
    scientific="incorrect",
    note="The answer describes a different article from patil25deep.pdf.",
)
```

No automatic LLM judge is introduced: the evaluation remains reproducible,
source-aware, and under the user's control.

In [ ]:
needs_review = [
    item for item in CASE_RESULTS if item["status"] in ("REVIEW", "ERROR")
]
for item in needs_review:
    print(
        f"{item['id']:<18} {item['status']:<7} "
        f"{', '.join(item.get('issues', []))}"
    )
print("Cases requiring review:", len(needs_review))

### 13.2 Save the results

Exports remain under this experiment's `outputs/notebook_evaluation` folder:

- `cases.json`: complete records, answers, links, model settings, and flags;
- `cases.csv`: compact comparison table;
- `manifest.json`: fixed configuration and conversation trace paths.

The original KíLín conversation JSON traces are preserved separately under
`logs/traces`. Keep the workspace when reporting regressions; a summary CSV
alone is insufficient to reproduce a routing or model failure.

In [ ]:
evaluation_dir = export_evaluation()

## 14. Next experimental iteration

When a mismatch occurs, determine **which layer** failed before changing code:

1. **Router:** wrong `intent`, `standalone`, or `related_turn_ids`.
2. **Policy/tool:** correct intent but unexpected retrieval mode, tool, or
   confirmation path.
3. **Scope:** correct route but wrong PDF(s) or previous result set.
4. **Model execution:** incomplete reasoning, token exhaustion, or failed
   tool generation.
5. **Scientific content:** correctly executed workflow, but incorrect paper
   identification, interpretation, or citation support.

Change one setting at a time (e.g., answering model **or** router model),
create new independent conversations, and preserve all traces and source
hashes. Only after identifying reproducible failures should the KíLín
implementation or intent descriptions be modified.